# Michigan Ave corridor speed (slow hours)

**Stop-level** Mansueto `speed_mph` (ping-interval speed), pooled across kept routes, for the **slowest hours** of each day slice.

- **Input:** `data/michigan_ave/segment_view.parquet` from [`michigan_ave_segment_view.ipynb`](michigan_ave_segment_view.ipynb).
- **Rows:** observed only (`interpolated == False`).
- **Exclusions:** Delaware, Chestnut (NB), Pearson (SB) — DLSD speed leak (vault `michigan-ave-slow-hour-corridor-plots`).
- **Charts:** latitude on x; **minutes/mile** on y (slow at top); mph on right via `secondary_yaxis`. Median, IQR band, dashed **p95 min/mi** (= `60 / p5(speed)`).
- **Day slices:** Mon–Thu, Friday, weekend (holidays not excluded; `cta-holiday-schedule-filtering`).

**QA (non-blocking):** ~137 backwards-time trips; PIDs **14130** / **14132** absent from segment view.

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.lines import Line2D

_nb = Path.cwd() if (Path.cwd() / "michigan_ave_segment.py").is_file() else Path.cwd() / "notebooks"
if str(_nb) not in sys.path:
    sys.path.insert(0, str(_nb))

from dlsd_run_quality import DEFAULT_PLOT_HOURS
from michigan_ave_segment import (
    corridor_hourly_median_speed,
    corridor_stop_min_per_mile_stats,
    corridor_street_tick_positions,
    direction_stop_sequences,
    filter_dlsd_leak_stops,
    load_kept_pids_meta,
    michigan_day_slice_mask,
    observed_speed_rows,
    shared_observation_window,
    SHARED_OBSERVATION_WINDOW_HOURS,
    slowest_hours,
)

REPO_ROOT = _nb.parent if _nb.name == "notebooks" else _nb
_style = REPO_ROOT / "notebooks" / "cta_analysis.mplstyle"
if _style.is_file():
    plt.style.use(_style)

SEGMENT_VIEW_PATH = REPO_ROOT / "data" / "michigan_ave" / "segment_view.parquet"
SPEED_TABLE_PATH = REPO_ROOT / "notebooks" / "michigan_ave_speed_plot_stops.csv"
PLOT_HOURS = list(DEFAULT_PLOT_HOURS)
MIN_OBS = 10
N_SLOW_HOURS = 3
SPEED_COLOR = "#9D8AB8"

DAY_SLICES = ["weekday", "friday", "weekend"]
DAY_SLICE_LABELS = {
    "weekday": "weekday (Mon–Thu)",
    "friday": "Friday",
    "weekend": "weekend (Sat–Sun)",
}
DIRECTIONS = ["Northbound", "Southbound"]

# Hours per slice that override the slow-hour auto-pick; set to None to use the auto-pick.
# Simply use 15, 16, 17 for all dates
SLOW_HOURS_OVERRIDE: dict[str, list[int]] | None = {
    "weekday": [15, 16, 17],
    "friday": [15, 16, 17],
    "weekend": [15, 16, 17],
}

In [ ]:
if not SEGMENT_VIEW_PATH.is_file():
    raise FileNotFoundError(
        f"Missing {SEGMENT_VIEW_PATH}. Run notebooks/michigan_ave_segment_view.ipynb first."
    )

view = pd.read_parquet(SEGMENT_VIEW_PATH)
obs_raw = observed_speed_rows(view)
obs, window_start, window_end = shared_observation_window(obs_raw)
pids_before = set(obs_raw["pid"].astype(str).unique())
pids_after = set(obs["pid"].astype(str).unique())
pids_dropped = sorted(pids_before - pids_after, key=lambda x: int(x))
obs_plot = filter_dlsd_leak_stops(obs)
seqs = direction_stop_sequences()
street_ticks = corridor_street_tick_positions(seqs)
meta = load_kept_pids_meta()

print(f"Segment view rows: {len(view):,}")
print(f"Observed rows with bus_stop_time: {len(obs_raw):,}")
print(
    f"Shared {SHARED_OBSERVATION_WINDOW_HOURS}h window: "
    f"{window_start} -> {window_end} ({len(obs):,} rows)"
)
if pids_dropped:
    print(f"PIDs with no rows in window ({len(pids_dropped)}): {', '.join(pids_dropped)}")
print(f"After DLSD-leak stop drop: {len(obs_plot):,}")
print(f"Distinct trips (in window): {obs['unique_trip_vehicle_day'].nunique():,}")
print(f"Cache variant: {meta.get('cache_variant', 'last30d')}")

## Slow-hour diagnostic

Corridor-wide median `speed_mph` by hour (DLSD-leak stops excluded). The **3 slowest hours** per day slice feed the corridor charts below, unless `SLOW_HOURS_OVERRIDE` (config cell) is set.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for day_slice in DAY_SLICES:
    hourly = corridor_hourly_median_speed(
        obs_plot, day_slice, hours=PLOT_HOURS, exclude_dlsd_leak=False
    )
    h = hourly["hour"].astype(int)
    med = hourly["median_mph"]
    m = med.notna()
    ax.plot(h[m], med[m], marker="o", label=DAY_SLICE_LABELS[day_slice])
ax.set_xlabel("Hour of day")
ax.set_ylabel("Corridor median speed (mph)")
ax.set_title("Michigan Ave — pooled median speed by hour (DLSD-leak stops excluded)")
ax.set_xticks(PLOT_HOURS[::2])
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

slow_hours_by_slice: dict[str, list[int]] = {}
rows = []
for day_slice in DAY_SLICES:
    if SLOW_HOURS_OVERRIDE and day_slice in SLOW_HOURS_OVERRIDE:
        chosen = sorted(SLOW_HOURS_OVERRIDE[day_slice])
    else:
        chosen = slowest_hours(
            obs_plot, day_slice, n=N_SLOW_HOURS, hours=PLOT_HOURS
        )
    slow_hours_by_slice[day_slice] = chosen
    rows.append({"day_slice": day_slice, "slow_hours": chosen})
print(f"Using {N_SLOW_HOURS} slowest hours per slice (or SLOW_HOURS_OVERRIDE):")
display(pd.DataFrame(rows))

## Corridor charts (slow hours)

One figure per day slice: northbound and southbound panels, stops ordered by latitude.

In [ ]:
def _masked_lat_xy(lats, values):
    y = np.asarray(values, dtype=float)
    x = np.asarray(lats, dtype=float)
    m = np.isfinite(y)
    return x[m], y[m]


def _apply_street_ticks(ax, ticks):
    if ticks is not None and not ticks.empty:
        ax.set_xticks(ticks["lat"])
        ax.set_xticklabels(ticks["label"], rotation=45, ha="right", fontsize=8)


def plot_corridor_min_per_mile(ax, stats_df, *, color=SPEED_COLOR):
    lat = stats_df["lat"].astype(float)
    q25, q75 = stats_df["q25_min_mi"], stats_df["q75_min_mi"]
    med, q95 = stats_df["median_min_mi"], stats_df["q95_min_mi"]
    x_lo, y_lo = _masked_lat_xy(lat, q25)
    x_hi, y_hi = _masked_lat_xy(lat, q75)
    if len(x_lo):
        ax.fill_between(x_lo, y_lo, y_hi, alpha=0.2, color=color)
    x_m, y_m = _masked_lat_xy(lat, med)
    if len(x_m):
        ax.plot(x_m, y_m, marker="o", linestyle="-", color=color)
    x_p, y_p = _masked_lat_xy(lat, q95)
    if len(x_p):
        ax.plot(x_p, y_p, linestyle="--", color=color, alpha=0.85)


def plot_slow_hour_corridor(obs, day_slice, slow_hours, seqs):
    slice_label = DAY_SLICE_LABELS[day_slice]
    hour_set = set(slow_hours)
    mask = michigan_day_slice_mask(obs, day_slice)
    sub_hours = obs.loc[mask & obs["hour"].isin(hour_set)]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
    for col, rtdir in enumerate(DIRECTIONS):
        ax = axes[col]
        sequence = seqs[rtdir]
        sub = sub_hours.loc[sub_hours["rtdir"] == rtdir]
        stats_df = corridor_stop_min_per_mile_stats(
            sub, sequence, min_obs=MIN_OBS
        )
        plot_corridor_min_per_mile(ax, stats_df)
        _apply_street_ticks(ax, street_ticks)
        ax.set_title(rtdir)
        ax.set_xlabel("Latitude (south → north)")
        ax.grid(True, alpha=0.3)
        secax = ax.secondary_yaxis(
            "right",
            functions=(lambda t: 60.0 / t, lambda m: 60.0 / m),
        )
        secax.set_ylabel("Speed (mph)")
        if col == 0:
            ax.set_ylabel("Minutes per mile")

    hour_str = ", ".join(str(h) for h in slow_hours)
    fig.suptitle(
        # f"Michigan Ave — {slice_label} — hours {hour_str}\n"
        # f"median (solid), IQR (band), p95 min/mi (dashed); n ≥ {MIN_OBS} per stop",
        f"Michigan Ave — {slice_label} — 3-6 pm\n"
        f"median (solid), IQR (band), p95 min/mi (dashed)",
        y=1.05,
        fontsize=12,
    )
    legend_handles = [
        Line2D([0], [0], color=SPEED_COLOR, marker="o", label="median"),
        Line2D([0], [0], color=SPEED_COLOR, alpha=0.2, linewidth=8, label="IQR"),
        Line2D([0], [0], color=SPEED_COLOR, linestyle="--", label="p95 min/mi"),
    ]
    fig.legend(handles=legend_handles, loc="upper right", fontsize=8)
    plt.tight_layout()
    return fig


for day_slice in DAY_SLICES:
    hours = slow_hours_by_slice[day_slice]
    if not hours:
        print(f"Skip {day_slice}: no slow hours identified")
        continue
    plot_slow_hour_corridor(obs_plot, day_slice, hours, seqs)
    plt.show()

## Trip counts behind corridor chart markers

One distribution per stop = pooled slow-hour rows for that day slice and direction. `n_obs` is what `MIN_OBS` gates; `plotted` matches whether the chart shows a marker.

In [ ]:
def _short_stop_title(stpnm: str) -> str:
    name = str(stpnm)
    prefix = "Michigan & "
    return name[len(prefix) :] if name.startswith(prefix) else name


for day_slice in DAY_SLICES:
    hours = slow_hours_by_slice.get(day_slice, [])
    if not hours:
        print(f"Skip {day_slice}: no slow hours")
        continue
    slice_label = DAY_SLICE_LABELS[day_slice]
    hour_set = set(hours)
    mask = michigan_day_slice_mask(obs_plot, day_slice)
    sub_hours = obs_plot.loc[mask & obs_plot["hour"].isin(hour_set)]
    rows: list[dict] = []
    for rtdir in DIRECTIONS:
        sequence = seqs[rtdir].sort_values("lat")
        sub = sub_hours.loc[sub_hours["rtdir"] == rtdir]
        for row in sequence.itertuples(index=False):
            stpid = str(row.stpid)
            stop_sub = sub.loc[sub["stpid"].astype(str) == stpid]
            valid = stop_sub.loc[stop_sub["speed_mph"].notna()]
            n_obs = int(len(valid))
            n_trips = int(valid["unique_trip_vehicle_day"].nunique()) if n_obs else 0
            rows.append(
                {
                    "rtdir": rtdir,
                    "stop": _short_stop_title(row.stpnm),
                    "stpid": stpid,
                    "lat": float(row.lat),
                    "n_trips": n_trips,
                    "n_obs": n_obs,
                    "plotted": n_obs >= MIN_OBS,
                }
            )
    counts = pd.DataFrame(rows).sort_values(["lat", "rtdir"], kind="mergesort")
    print(f"\n{slice_label} — slow hours {list(hours)}")
    display(counts)

## Save plotted data

One row per day slice × direction × corridor stop, written to `notebooks/michigan_ave_speed_plot_stops.csv`. The `*_min_mi` columns are what the corridor charts plot; stops with `plotted == False` (n < `MIN_OBS`) have blank stats.

In [ ]:
STAT_COLUMNS = [
    "median_min_mi", "q25_min_mi", "q75_min_mi", "q95_min_mi",
    "median_mph", "q25_mph", "q75_mph", "q05_mph",
]


def corridor_speed_table(obs, day_slice, slow_hours, seqs):
    mask = michigan_day_slice_mask(obs, day_slice)
    sub_hours = obs.loc[mask & obs["hour"].isin(set(slow_hours))]
    parts = []
    for rtdir in DIRECTIONS:
        sequence = seqs[rtdir].sort_values("seq_index")
        sub = sub_hours.loc[sub_hours["rtdir"] == rtdir]
        stats_df = corridor_stop_min_per_mile_stats(sub, sequence, min_obs=MIN_OBS)
        valid = sub.loc[sub["speed_mph"].notna()]
        n_trips = valid.groupby(valid["stpid"].astype(str))["unique_trip_vehicle_day"].nunique()
        stats_df = stats_df.rename(columns={"n": "n_obs"})
        stats_df["rtdir"] = rtdir
        stats_df["seq_index"] = sequence["seq_index"].to_numpy()
        stats_df["n_trips"] = stats_df["stpid"].map(n_trips).fillna(0).astype(int)
        stats_df["plotted"] = stats_df["n_obs"] >= MIN_OBS
        parts.append(stats_df)
    out = pd.concat(parts, ignore_index=True)
    out["day_slice"] = day_slice
    out["hours"] = ";".join(str(h) for h in slow_hours)
    return out


speed_table = pd.concat(
    [
        corridor_speed_table(obs_plot, day_slice, slow_hours_by_slice[day_slice], seqs)
        for day_slice in DAY_SLICES
        if slow_hours_by_slice.get(day_slice)
    ],
    ignore_index=True,
)
speed_table["window_start"] = window_start
speed_table["window_end"] = window_end
speed_table["min_obs"] = MIN_OBS
speed_table[STAT_COLUMNS] = speed_table[STAT_COLUMNS].round(3)
speed_table = speed_table[
    [
        "day_slice", "hours", "rtdir", "seq_index", "stpid", "stpnm", "lat",
        "n_obs", "n_trips", "plotted", *STAT_COLUMNS,
        "window_start", "window_end", "min_obs",
    ]
]
speed_table.to_csv(SPEED_TABLE_PATH, index=False)
print(f"Wrote {len(speed_table):,} rows to {SPEED_TABLE_PATH}")
display(speed_table.head())